In [1]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/train.csv')
irrigation.head()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, recall_score,  accuracy_score
# splitting data into train and test sets
X = irrigation.drop(['Irrigation_Need', 'id'], axis=1)
y = irrigation['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42) 



In [6]:
from sklearn.preprocessing import LabelEncoder

# encode target labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# convert object columns to category for XGBoost
categorical_cols = [
    'Soil_Type',
    'Crop_Type',
    'Crop_Growth_Stage',
    'Season',
    'Irrigation_Type',
    'Water_Source',
    'Mulching_Used',
    'Region'
]

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# training xgboost model
xgb = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    enable_categorical=True,
    tree_method='hist'
)
xgb.fit(X_train, y_train)

# evaluating xgboost model
y_pred_xgb = xgb.predict(X_test)

# convert predictions back to original labels
y_pred_xgb = le.inverse_transform(y_pred_xgb)
y_test = le.inverse_transform(y_test)

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:24:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.97      0.97    126000
weighted avg       0.99      0.99      0.99    126000

XGBoost Accuracy: 0.9853492063492063


In [8]:
from sklearn.preprocessing import LabelEncoder

# encode full target for cross-validation
le = LabelEncoder()
y = le.fit_transform(y)

# make sure X has categorical dtypes (no loops)
X = X.astype({
    'Soil_Type': 'category',
    'Crop_Type': 'category',
    'Crop_Growth_Stage': 'category',
    'Season': 'category',
    'Irrigation_Type': 'category',
    'Water_Source': 'category',
    'Mulching_Used': 'category',
    'Region': 'category'
})

# run cross-validation
xgb_cv_scores = cross_val_score(xgb, X, y, cv=5)
print("XGBoost CV Scores:", xgb_cv_scores)
print("XGBoost CV Mean Score:", np.mean(xgb_cv_scores))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:32:45] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:32:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:32:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:33:07] WARN

XGBoost CV Scores: [0.98481746 0.98463492 0.98476984 0.98379365 0.98397619]
XGBoost CV Mean Score: 0.9843984126984127


In [9]:
xgb_cv_recall = cross_val_score(xgb, X, y, cv=5, scoring='recall')
print("XGBoost CV Recall Scores:", xgb_cv_recall)
print("XGBoost CV Mean Recall Score:", np.mean(xgb_cv_recall))
print ("XGBoost classification report with recall as evaluation metric:")
print(classification_report(y_test, y_pred_xgb))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:34:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:971: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 152, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 408, in _score
    return self._sign * self._score_func(y_true, y_pr

XGBoost CV Recall Scores: [nan nan nan nan nan]
XGBoost CV Mean Recall Score: nan
XGBoost classification report with recall as evaluation metric:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.97      0.97    126000
weighted avg       0.99      0.99      0.99    126000



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:971: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 152, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 408, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "/Libr

In [11]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_weighted = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    enable_categorical=True,
    tree_method='hist'
)

xgb_weighted.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb_weighted = xgb_weighted.predict(X_test)
print("XGBoost (Weighted) Classification Report:")
print(classification_report(y_test, y_pred_xgb_weighted))
print("XGBoost (Weighted) Accuracy:", accuracy_score(y_test, y_pred_xgb_weighted))


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:39:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost (Weighted) Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      4202
           1       0.99      1.00      0.99     73983
           2       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.96      0.97      0.97    126000
weighted avg       0.99      0.98      0.98    126000

XGBoost (Weighted) Accuracy: 0.9849682539682539


In [12]:
#hyperparameter tuning for xgboost adjusted for class imbalance with 2 hyperparameters and 2 values each
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6]
}
xgb_grid_search = GridSearchCV(estimator=xgb_weighted, param_grid=xgb_param_grid, cv=5, scoring='recall')
xgb_grid_search.fit(X_train, y_train)
print("Best XGBoost Hyperparameters:", xgb_grid_search.best_params_)
best_xgb = xgb_grid_search.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)
print("Best XGBoost Classification Report:")
print(classification_report(y_test, y_pred_best_xgb))
print("Best XGBoost Accuracy:", accuracy_score(y_test, y_pred_best_xgb))    


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [10:40:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwarg

Best XGBoost Hyperparameters: {'max_depth': 3, 'n_estimators': 100}
Best XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000

Best XGBoost Accuracy: 0.9851031746031746


In [14]:
import pandas as pd

# 1. Load the test dataset
test_data = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/test.csv')

# Save test IDs and drop the column from features
test_ids = test_data['id']
X_test = test_data.drop(['id'], axis=1) 

# 2. Force the test set to have the EXACT same data types (including categories) as your X_train
# This ensures XGBoost is happy without running get_dummies!
X_test_aligned = X_test.astype(X_train.dtypes.to_dict())

# 3. Predict using your trained XGBoost model
xgb_predictions = xgb.predict(X_test_aligned)

# 4. Convert numeric predictions (0, 1, 2) back to original text ('High', 'Low', 'Medium')
reverse_mapping = {0: 'High', 1: 'Low', 2: 'Medium'}
final_text_predictions = pd.Series(xgb_predictions).map(reverse_mapping)

# 5. Create final DataFrame and save to CSV
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': final_text_predictions
})

# Save to your Homework-2 folder without the index numbers!
submission.to_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/xgboost_submission.csv', index=False)

print("Your file is perfectly ready! Use Kaggle to submit 'xgboost_submission.csv'")


Your file is perfectly ready! Use Kaggle to submit 'xgboost_submission.csv'
